# Crivo: hipóteses e fontes de fatos

Este caderno carrega os pesos próprios já treinados. Você não precisa repetir o treino nem contratar GPU para inspecioná-los. O candidato é experimental: os testes e seus erros estão conservados abaixo, sem troca automática dos pesos do chat.

A rodada ensina a marcar se uma situação é hipótese e selecionar qual fonte continua válida após correção, abandono ou confirmação. Usa interpretação neural, cópia literal limitada e operações fechadas de preços/requisitos. Não recebeu treino de redação livre.


In [ ]:
from pathlib import Path
import subprocess,sys,json
REPO=Path('/content/CRIVO-memoria')
BRANCH='codex/memoria-fontes-20261009'
if not REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/HROSONE/CRIVO.git',str(REPO)])
else:
    b=subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()
    if b!=BRANCH:raise RuntimeError('Pasta de outra branch; escolha uma pasta nova.')
print('Revisão:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
LAB=REPO/'experimentos/memoria_fontes_20261009'


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.2,<3','numpy>=1.24,<3','tokenizers>=0.20,<1'])


## Resultados e limites

O painel reservado foi escrito por outro agente e aberto somente depois dos dois treinos e da escolha pelo dev. Ainda é autoria da mesma equipe e gramática finita, não uma avaliação externa de conversa livre.

Os pesos anteriores são medidos com preparo antigo e novo, para separar a correção da ordem dos nomes da adaptação neural. Os dois candidatos usam os mesmos lotes; somente a supervisão auxiliar dos bancos de fontes difere. O corpo e as cabeças anteriores permanecem congelados, mas as novas cabeças podem modificar o comportamento e precisam passar pelas regressões.


**Resultado: ambos os novos checkpoints foram reprovados.** O anterior acertou 247/408, o controle 229/408 e fontes 240/408. A seleção pelo dev serve à inspeção do experimento, não recomenda substituir os pesos anteriores. Correções e confirmações pioraram; todos os erros permanecem publicados.


In [ ]:
r=json.loads((LAB/'avaliacao/resumo.json').read_text())
for nome,paineis in r['resultados'].items():
    m=paineis['prospectivo_autoria_limitado']
    print(nome,m['contratos_corretos'],'/',m['n'],'sessões completas:',m['sessoes_completas'],'/',m['sessoes'])
    print({k:(x['contratos_corretos'],x['n']) for k,x in m['por_evento'].items()})
print('Critérios:',r['criterios'])
print('Limites:',r['limites'])
SELECAO=json.loads((LAB/'escolha_pelo_dev.json').read_text())
print('Escolha anterior ao teste:',SELECAO)


## Inspecione suas falas

A versão padrão é a que foi escolhida pela validação antes do teste. Você pode usar `MODELO='anterior'`, `'controle'` ou `'fontes'` para comparar. O modo `anterior` também usa o preparo novo e corresponde a `anterior_preparo_novo` no resumo. Sua cabeça de evento não foi treinada e aparece sem previsão útil. As fontes na saída apontam para o histórico real. O evento previsto não é o evento correto de referência: as decisões vêm da rede.

Nomes de uma palavra com inicial maiúscula, até oito candidatos e no máximo 256 tokens, sem truncamento. Um cálculo executável pode usar uma fonte errada. As falas editadas ficam em um arquivo temporário da sessão, sem envio automático para treino ou Drive. Esta célula não redige conversa livre.


**Falha observada na inspeção local:** na sequência abaixo, após confirmar 19 reais como fato, o último retorno ainda escolhe 41 reais. Os quatro primeiros passos estão corretos; o quinto está errado. O checkpoint anterior com o mesmo preparo também erra esse passo. O traço completo está em `verificacao_caderno.json`; não confunda execução numérica válida com manutenção correta do fato confirmado.


In [ ]:
MODELO=SELECAO['braco']
FALAS=[
    'O plano Azul custa 23 reais. O plano Verde custa 35 reais. Compare Azul e Verde.',
    'Corrigindo Azul: o custo real é 41 reais. Compare Azul e Verde.',
    'Imagine Azul por 19 reais, sem alterar os fatos. Compare Azul e Verde.',
    'A última hipótese aconteceu de verdade. Agora ela é fato. Compare Azul e Verde.',
    'Esqueça a simulação e use os preços reais corrigidos. Compare Azul e Verde.',
]
p=Path('/content/falas-memoria.json');p.write_text(json.dumps(FALAS,ensure_ascii=False))
RUNNER="""
import sys,json,torch
from pathlib import Path
sys.path.insert(0,sys.argv[1])
from apoio import ROOT,INICIAL
from rede import carregar
from preparo import codificar
from modelo import lote,executar
from decodificar import proposta_limitada
from dados import EVENTOS
from tokenizers import Tokenizer
torch.set_num_threads(1)
tok=Tokenizer.from_file(str(ROOT/'artefatos/linguagem_profunda/tokenizer.json'));tok.encode_special_tokens=True
cp=INICIAL if sys.argv[3]=='anterior' else Path(sys.argv[1])/sys.argv[3]/'pesos.pt'
m,_=carregar(tok,path=cp);m.eval()
falas=json.loads(Path(sys.argv[2]).read_text())
for i in range(1,len(falas)+1):
    e={'turnos':falas[:i],'argumentos':[None]*3,'referente':None}
    it=codificar(tok,e);x,l=lote([it],tok.token_to_id('<pad>'))
    with torch.no_grad():ls=m(x,l)
    pr=proposta_limitada(it,ls,0)
    print(json.dumps({'turno':i,'evento_previsto':None if sys.argv[3]=='anterior' else EVENTOS[int(ls['evento'][0].argmax())],
        'evento_turno_sem_treino':sys.argv[3]=='anterior',
        'proposta':pr,'execucao':executar(pr)},ensure_ascii=False))
"""
subprocess.check_call([sys.executable,'-c',RUNNER,str(LAB),str(p),MODELO])


## Treinamento concluído

A rede tem aproximadamente 3,08 milhões de parâmetros. Nesta rodada foram ajustados cerca de 136 mil parâmetros de cabeças próprias, mantendo o corpo anterior congelado. O mesmo corpo lê o histórico e a última fala, sem modelo externo. O arquivo de escolha do dev identifica o checkpoint, não uma aprovação para o chat.

Os scripts e dados permitem reprodução; checkpoints completos de Adam/RNG ficam no laboratório local. Este caderno não inicia treino. A inspeção foi validada localmente em CPU; não foi executada numa sessão real do Colab.
